In [1]:
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import torch
import sacrebleu
from tqdm import tqdm

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load translation model and tokenizer
def load_translation_model(model_name):
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    return tokenizer, model

In [3]:
# Translate a sentence or batch
def translate(texts, tokenizer, model):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]

In [4]:
# Load models
en2cy_tokenizer, en2cy_model = load_translation_model("Helsinki-NLP/opus-mt-en-cy")
cy2en_tokenizer, cy2en_model = load_translation_model("Helsinki-NLP/opus-mt-cy-en")

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\transformers\models\marian\tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [5]:
# Load SentenceTransformer model
sim_model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\c24082331\.cache\huggingface\hub\models--sentence-transformers--distiluse-base-multilingual-cased-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet S

In [6]:
sim_model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: DistilBertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 512, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
)

In [11]:
#Semantic similarity
def semantic_similarity(s1, s2):
    emb1 = sim_model.encode(s1, convert_to_tensor=True)
    emb2 = sim_model.encode(s2, convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()

In [12]:
def evaluate(original, back_translated):
    bleu = sacrebleu.corpus_bleu([back_translated], [[original]]).score
    chrf = sacrebleu.corpus_chrf([back_translated], [[original]]).score
    cosine = semantic_similarity(original, back_translated)
    return round(bleu, 2), round(chrf, 2), round(cosine, 4)

In [8]:
#English UniversalCEFR datasets
english_datasets = [
    load_dataset("UniversalCEFR/readme_en")["train"],
    load_dataset("UniversalCEFR/cefr_asag_en")["train"],
    load_dataset("UniversalCEFR/icle500_en")["train"],
    load_dataset("UniversalCEFR/cefr_sp_en")["train"],
    load_dataset("UniversalCEFR/elg_cefr_en")["train"],
    load_dataset("UniversalCEFR/cambridge_exams_en")["train"],
]

# Combine datasets
english_data = concatenate_datasets(english_datasets)

# Filter for A1 and A2 examples only
english_a1_a2 = english_data.filter(lambda example: example["cefr_level"] in ["A1", "A2"])

df = english_a1_a2.to_pandas()[["text", "cefr_level"]].dropna().reset_index(drop=True)

In [14]:
english_datasets

[Dataset({
     features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
     num_rows: 2822
 }),
 Dataset({
     features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
     num_rows: 299
 }),
 Dataset({
     features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
     num_rows: 495
 }),
 Dataset({
     features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
     num_rows: 10004
 }),
 Dataset({
     features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
     num_rows: 712
 }),
 Dataset({
     features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
     num_rows: 331
 })]

In [9]:
# Count how many samples are labeled A1 and A2
df["cefr_level"].value_counts()

cefr_level
A2    2178
A1     348
Name: count, dtype: int64

In [10]:
# Back-translation loop
records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Back-translating"):
    original_en = row["text"]
    cefr = row["cefr_level"]
    
    try:
        #English to Welsh
        welsh = translate([original_en], en2cy_tokenizer, en2cy_model)[0]

        #Welsh to English (back-translation)
        back_en = translate([welsh], cy2en_tokenizer, cy2en_model)[0]

        #Evaluate
        bleu, chrf, cosine = evaluate(original_en, back_en)

        records.append({
            "original_english": original_en,
            "translated_welsh": welsh,
            "back_translated_english": back_en,
            "cefr_level": cefr,
            "BLEU": bleu,
            "chrF": chrf,
            "cosine_similarity": cosine
        })

    except Exception as e:
        print(f"Error on: {original_en}\n{e}")

# Final DataFrame
df_bt = pd.DataFrame(records)

Back-translating: 100%|██████████| 2526/2526 [2:01:32<00:00,  2.89s/it]  


In [13]:
df_bt

,original_english,translated_welsh,back_translated_english,cefr_level,BLEU,chrF,cosine_similarity
0,The late 19th century marks the start of psych...,Ym 19egrodd y bedwaredd ganrif ar bymtheg o bw...,In 19th century the late 1800 ' s of the late ...,A2,6.92,26.57,0.4495
1,Cartridge paper is the basic type of drawing p...,Constellation name (optional),FIVE FEUDAL LORDS,A2,0.00,0.00,-0.0948
2,This is called the observer effect.,Mae hyn yn enw' r effaith.,This is the name of the effect.,A2,16.52,35.91,0.7631
3,This was originally intended to provide a chec...,Roedd hyn yn bwriadu darparu grym gwleidyddol ...,This meant to provide political power on polit...,A2,26.58,47.73,0.7600
4,Different policies were applied in Albania and...,Cafodd pob paid a oedd yn fwy o'r Undeb Sofiet...,All of the Soviet Union made more the Soviet U...,A2,15.22,24.89,0.4676
...,...,...,...,...,...,...,...
2521,"The Elephant Show \n\nby Daniel Allsop, age 14...","Mae'r hawyr yn gwisgo gan Daniel bob oedran, 1...","The haus was dressed by Daniel's age, 14, I we...",A2,0.07,7.90,0.3191
2522,Mount Kilimanjaro \n\nMount Kilimanjaro is the...,(2 Bren. 15: 6 - 8) Er bod Mynyddmanman Americ...,Although Mountmanman America were beyond the h...,A2,0.03,9.62,0.4589
2523,"Visit the Edinburgh Festival!\n\nEvery year, t...","Ewch i'r ddesbys, Bob flwyddyn, mae miloedd o ...","Go to the last year, Each year during this yea...",A2,0.10,7.64,0.2307
2524,The Rhino\n\nThere are five different types of...,Mae'r cannoedd o bump yn cynnwys mathau gwahan...,The hundreds of five types of different types ...,A2,0.05,11.47,0.2665


In [12]:
# Filter records with cosine similarity > 0.80
filtered_df = df_bt[df_bt["cosine_similarity"] > 0.80]

# Save the filtered DataFrame to a CSV file
filtered_df.to_csv("high_cosine_similarity_records.csv", index=False)